# =============== Error Correction Model ===============

In [22]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint

In [23]:
price_files = [
    "/Users/jameszhao/IMC3/Pre-Comp/Data1/prices_round_1_day_-2.csv",
    "/Users/jameszhao/IMC3/Pre-Comp/Data1/prices_round_1_day_-1.csv",
    "/Users/jameszhao/IMC3/Pre-Comp/Data1/prices_round_1_day_0.csv"
]

# Read and concatenate all price files
price_dfs = [pd.read_csv(f, sep=',') for f in price_files]
all_prices_df = pd.concat(price_dfs, ignore_index=True)

# Save to a new combined CSV
combined_price_path = "/Users/jameszhao/IMC3/Pre-Comp/Data1/prices_combined_days_-2_to_0.csv"
df = pd.read_csv(combined_price_path, sep=',')

In [24]:
products = ['RAINFOREST_RESIN', 'KELP', 'SQUID_INK']

In [25]:
kelp = df[df['product'] == 'KELP'].groupby('timestamp')['mid_price'].mean()
squid = df[df['product'] == 'SQUID_INK'].groupby('timestamp')['mid_price'].mean()

In [26]:
data = pd.concat([kelp, squid], axis=1)
data.columns = ['KELP', 'SQUID_INK']
data.dropna(inplace=True)

In [27]:
print("ADF p-value for KELP:", adfuller(data['KELP'])[1])
print("ADF p-value for SQUID_INK:", adfuller(data['SQUID_INK'])[1])
print("Cointegration test p-value:", coint(data['KELP'], data['SQUID_INK'])[1])

ADF p-value for KELP: 0.8133661939984123
ADF p-value for SQUID_INK: 0.2184450575194274
Cointegration test p-value: 0.8578837721552339


In [28]:
coint_model = sm.OLS(data['KELP'], sm.add_constant(data['SQUID_INK'])).fit()
residuals = coint_model.resid

In [29]:
data['D_KELP'] = data['KELP'].diff()
data['D_SQUID_INK'] = data['SQUID_INK'].diff()
data['resid_lag'] = residuals.shift(1)
data.dropna(inplace=True)

In [30]:
X = sm.add_constant(data[['D_SQUID_INK', 'resid_lag']])
y = data['D_KELP']
ecm_result = sm.OLS(y, X).fit()

In [31]:
print(ecm_result.summary())

                            OLS Regression Results                            
Dep. Variable:                 D_KELP   R-squared:                       0.103
Model:                            OLS   Adj. R-squared:                  0.103
Method:                 Least Squares   F-statistic:                     574.7
Date:                Mon, 07 Apr 2025   Prob (F-statistic):          5.49e-237
Time:                        18:40:23   Log-Likelihood:                -5945.3
No. Observations:                9999   AIC:                         1.190e+04
Df Residuals:                    9996   BIC:                         1.192e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.0018      0.004      0.413      